# Fine-tune only the Hugging Face ViT on multi-generator GenImage

**Run in Colab with a GPU (T4 or better).** This notebook fine-tunes only the ViT; it does not load or train the CNN. It starts from the ViT weights in `ai_vs_real_genimage_outputs_v3.zip` and writes v4 outputs separately so previous checkpoints remain intact.

Training uses a balanced subset from multiple available generators (default: BigGAN, VQDM, Wukong, GLIDE, Stable Diffusion 1.4 and 1.5). ADM is reserved for validation/checkpoint selection. Midjourney is kept out of training and validation for the final held-out test. Real/nature = 0; generated/AI = 1. Exact-byte duplicates are filtered across splits.

The training data scale and generators are configurable below. More diverse sources may improve transfer, but **90% accuracy is not guaranteed**. Do not select settings based on the held-out test score. GenImage is a research benchmark: review its terms, cite its authors, and do not redistribute the dataset.

**Steps:** mount Drive, provide the shared GenImage folder and `ai_vs_real_genimage_outputs_v3.zip`, run all cells, then review the ADM validation and Midjourney test metrics before using the exported ViT.


## 1. Install dependencies, configure the ViT run, and select a device

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.46,<5", "safetensors", "scikit-learn", "pandas", "pillow", "tqdm"],
    check=True,
)


In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path, PurePosixPath

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup,
)
from google.colab import drive, files

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
TRAIN_PER_CLASS = 50_000
VALIDATION_PER_CLASS = 5_000
TEST_PER_CLASS = 5_000
TRAIN_GENERATORS = [
    "BigGAN",
    "VQDM",
    "wukong",
    "glide",
    "stable_diffusion_v_1_4",
    "stable_diffusion_v_1_5",
]
REAL_TRAIN_GENERATOR = "BigGAN"
VALIDATION_GENERATOR = "ADM"
TEST_GENERATOR = "Midjourney"
CLASS_FOLDERS = {0: "nature", 1: "ai"}

VIT_BATCH_SIZE = 16
VIT_EPOCHS = 5
VIT_LEARNING_RATE = 1e-5
VIT_WEIGHT_DECAY = 0.01
VIT_GRAD_ACCUMULATION_STEPS = 2
VIT_EARLY_STOPPING_PATIENCE = 2
VIT_MAX_GRAD_NORM = 1.0

FORCE_RETRAIN = False
DOWNLOAD_ZIP_AT_END = True
DRIVE_GENIMAGE_OVERRIDE = None
V3_ZIP_NAME = "ai_vs_real_genimage_outputs_v3.zip"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | ViT device: {DEVICE}")
if DEVICE.type != "cuda":
    raise RuntimeError("A Colab GPU is required for this ViT training run. Select a T4 or better runtime.")
print(f"Free local disk: {shutil.disk_usage('/content').free / 1e9:.1f} GB")


## 2. Mount Drive, find GenImage, and load the v3 ViT checkpoint

Add a shortcut to the dataset owner's shared GenImage folder in My Drive (or copy the needed archives). The root must contain generator folders for every name in `TRAIN_GENERATORS`, plus ADM and Midjourney. Upload the v3 ZIP if the notebook cannot find it in Drive. Only the ViT checkpoint and image processor are extracted; CNN files are ignored.


In [ ]:
drive.mount("/content/drive")
MYDRIVE = Path("/content/drive/MyDrive")
if not MYDRIVE.is_dir():
    raise RuntimeError("Google Drive is not mounted. Rerun this cell and approve Drive access.")

REQUIRED_GENERATORS = list(dict.fromkeys(TRAIN_GENERATORS + [VALIDATION_GENERATOR, TEST_GENERATOR]))


def child_dir(parent, name):
    try:
        for candidate in parent.iterdir():
            if candidate.is_dir() and candidate.name.casefold() == name.casefold():
                return candidate
    except OSError as error:
        raise RuntimeError(f"Cannot inspect Drive folder {parent}: {error}") from error
    return None


def discover_genimage_root():
    if DRIVE_GENIMAGE_OVERRIDE:
        candidates = [Path(DRIVE_GENIMAGE_OVERRIDE)]
    else:
        candidates = [MYDRIVE, MYDRIVE / "Colab Notebooks"]
        shortcut_targets = Path("/content/drive/.shortcut-targets-by-id")
        if shortcut_targets.is_dir():
            candidates.extend(item for item in shortcut_targets.iterdir() if item.is_dir())
        for candidate in list(candidates):
            if candidate.is_dir():
                genimage = child_dir(candidate, "genimage")
                if genimage is not None:
                    candidates.append(genimage)
    for candidate in candidates:
        if candidate.is_dir() and all(child_dir(candidate, name) for name in REQUIRED_GENERATORS):
            return candidate
    raise FileNotFoundError(
        f"Could not find a GenImage root containing {REQUIRED_GENERATORS}. "
        "Add a shortcut to the shared folder in My Drive, or set DRIVE_GENIMAGE_OVERRIDE."
    )


DRIVE_GENIMAGE = discover_genimage_root()
GEN_DIRS = {name: child_dir(DRIVE_GENIMAGE, name) for name in REQUIRED_GENERATORS}
print(f"GenImage root: {DRIVE_GENIMAGE}")
for name, folder in GEN_DIRS.items():
    print(f"  {name}: {folder}")

OUTPUT_DIR = Path("/content/ai_vs_real_vit_genimage_outputs_v4")
DRIVE_OUT = MYDRIVE / "ai_vs_real_vit_genimage_outputs_v4"
VIT_OUTPUT_DIR = OUTPUT_DIR / "huggingface_vit"
METRICS_DIR = OUTPUT_DIR / "metrics"
SPLITS_DIR = OUTPUT_DIR / "splits"
VIT_STAGE_FILE = METRICS_DIR / "vit_stage.json"
VIT_SCORES_FILE = METRICS_DIR / "vit_test_scores.npy"
VIT_TRUE_FILE = METRICS_DIR / "vit_test_true.npy"

if FORCE_RETRAIN and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
if not FORCE_RETRAIN and DRIVE_OUT.is_dir() and not OUTPUT_DIR.exists():
    print(f"Restoring previous v4 run from {DRIVE_OUT}")
    shutil.copytree(DRIVE_OUT, OUTPUT_DIR, ignore=shutil.ignore_patterns("*.zip"))
for folder in (VIT_OUTPUT_DIR, METRICS_DIR, SPLITS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def sync_to_drive():
    shutil.copytree(OUTPUT_DIR, DRIVE_OUT, dirs_exist_ok=True)
    print(f"Saved progress to {DRIVE_OUT}")


PREV_DIR = Path("/content/previous_v3_outputs")
VIT_CHECKPOINT_DIR = PREV_DIR / "huggingface_vit"
V3_KEEP_DIR = MYDRIVE / "AI_vs_Real"
V3_OUTPUT_DIR = MYDRIVE / "ai_vs_real_genimage_outputs_v3"


def v3_vit_ready():
    required = ("config.json", "model.safetensors", "preprocessor_config.json")
    return all((VIT_CHECKPOINT_DIR / filename).is_file() for filename in required)


def find_v3_zip():
    for folder in (V3_KEEP_DIR, V3_OUTPUT_DIR, MYDRIVE, MYDRIVE / "Colab Notebooks", DRIVE_GENIMAGE):
        candidate = folder / V3_ZIP_NAME
        if candidate.is_file():
            return candidate
    return None


def extract_v3_vit(zip_path):
    PREV_DIR.mkdir(parents=True, exist_ok=True)
    root = PREV_DIR.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue
            parts = PurePosixPath(member.filename).parts
            if PurePosixPath(member.filename).is_absolute() or ".." in parts:
                raise ValueError(f"Unsafe path in v3 checkpoint ZIP: {member.filename}")
            index = next((i for i, part in enumerate(parts) if part == "huggingface_vit"), None)
            if index is None:
                continue
            target = (PREV_DIR / Path(*parts[index:])).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe model path in ZIP: {member.filename}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)


if not v3_vit_ready():
    v3_zip = find_v3_zip()
    if v3_zip is None:
        print(f"Upload {V3_ZIP_NAME}; only its ViT model and processor will be used.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise ValueError(f"Upload exactly one v3 checkpoint ZIP; received {list(uploaded)}")
        v3_zip = Path("/content") / zip_names[0]
        V3_KEEP_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(v3_zip, V3_KEEP_DIR / V3_ZIP_NAME)
    print(f"Using v3 checkpoint archive: {v3_zip}")
    extract_v3_vit(v3_zip)
if not v3_vit_ready():
    raise FileNotFoundError(f"The v3 ZIP does not contain a complete ViT checkpoint: {VIT_CHECKPOINT_DIR}")
print(f"Starting ViT checkpoint: {VIT_CHECKPOINT_DIR}")


## 3. Extract only the required generator images

ADM and Midjourney validation folders are used for their respective splits. Training AI images are drawn evenly across all configured training generators; real/nature training images are drawn from BigGAN's shared real-image folder. Only the required subsets are extracted to Colab local disk. The helper supports the multi-volume ZIP archives in the shared GenImage folder and reuses completed extractions after a runtime restart.


In [ ]:
LOCAL_ROOT = Path("/content/genimage_extracted")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
SEVENZ = shutil.which("7z")

def find_master_zip(gdir):
    zips = sorted(p for p in gdir.iterdir() if p.is_file() and p.suffix.lower() == ".zip")
    for z in zips:                                     # prefer the one that has .z01 siblings
        if z.with_suffix(".z01").exists():
            return z
    return zips[0] if zips else None

def index_archive(archive, wanted):
    """List an archive once. Returns {(split,label): (internal_prefix, [member names])} for wanted pairs."""
    err = Path("/content/7z_list.err")
    found, prefixes = defaultdict(list), {}
    with err.open("w") as ef:
        proc = subprocess.Popen([SEVENZ, "l", "-slt", "-ba", str(archive)], stdout=subprocess.PIPE,
                                stderr=ef, text=True, errors="replace")
        cur, is_dir = None, False
        def flush():
            if not cur or is_dir or Path(cur).suffix.lower() not in IMAGE_EXTENSIONS:
                return
            parts = cur.split("/")
            for i in range(len(parts) - 2):
                key = (parts[i], parts[i + 1])
                if key in wanted:
                    prefix = "/".join(parts[:i])
                    if key not in prefixes:
                        prefixes[key] = prefix
                    if prefixes[key] == prefix:
                        found[key].append(cur)
                    break
        for line in proc.stdout:
            line = line.rstrip("\n")
            if line.startswith("Path = "):
                flush(); cur, is_dir = line[7:].replace("\\", "/"), False
            elif line.startswith("Folder = "):
                is_dir = line.endswith("+")
        flush()
        proc.wait()
    if proc.returncode not in (0, 1) and not found:
        raise RuntimeError(f"7z could not list {archive}:\n{err.read_text()[-1500:]}")
    return {k: (prefixes[k], v) for k, v in found.items()}

def build_patterns(prefix, split, label, names, need):
    """7z include patterns. If need is None (or the folder is small) take everything; otherwise take an
    evenly spread subset by last character of the file name (>= need*1.15 files)."""
    base = f"{prefix}/{split}/{label}" if prefix else f"{split}/{label}"
    if need is None or len(names) <= need * 1.3:
        return [f"{base}/*"], len(names)
    groups = defaultdict(int)
    for n in names:
        p = Path(n); groups[(p.stem[-1:], p.suffix)] += 1
    keys = sorted(groups); random.Random(SEED).shuffle(keys)
    picked, total = [], 0
    for k in keys:
        picked.append(k); total += groups[k]
        if total >= need * 1.15:
            break
    return [f"{base}/*{c}{e}" for c, e in picked], total

def run_7z(archive, out_dir, patterns, class_dir, expected, tag):
    log = Path(f"/content/7z_{tag}.log")
    for attempt in range(1, 4):
        with log.open("w") as lf:
            proc = subprocess.Popen([SEVENZ, "x", "-y", "-bso0", "-bsp0", str(archive), f"-o{out_dir}", *patterns],
                                    stdout=lf, stderr=subprocess.STDOUT)
            t0 = time.time(); last = 0.0
            while proc.poll() is None:
                time.sleep(2)
                if time.time() - last > 30:
                    last = time.time()
                    n = sum(1 for _ in class_dir.iterdir()) if class_dir.is_dir() else 0
                    print(f"   ... {n:,}/{expected:,} files, {(time.time()-t0)/60:.1f} min", flush=True)
        if proc.returncode in (0, 1):
            return
        print(f"   7z failed (exit {proc.returncode}), attempt {attempt}/3: {log.read_text()[-400:]}")
        time.sleep(10)
    raise RuntimeError(f"7z extraction failed for {tag}. See {log}")

ARCHIVE_INDEX, CLASS_DIRS = {}, {}

def prepare_generator(gen, jobs):
    """jobs: list of (split, label, need_or_None). Fills CLASS_DIRS[(gen,split,label)]."""
    gdir = GEN_DIRS[gen]
    pending = []
    for split, label, need in jobs:
        marker = LOCAL_ROOT / gen / f".done_{split}_{label}"
        if marker.is_file() and Path(marker.read_text().strip()).is_dir():
            CLASS_DIRS[(gen, split, label)] = Path(marker.read_text().strip())
            print(f"[Skipped] {gen}/{split}/{label} already extracted")
        else:
            pending.append((split, label, need))
    if not pending:
        return
    archive = find_master_zip(gdir)
    if archive is None:
        # fallback: the folder on Drive may already contain extracted images
        for split, label, _ in pending:
            hits = [p for p in list(gdir.glob(f"{split}/{label}")) + list(gdir.glob(f"*/{split}/{label}")) if p.is_dir()]
            if not hits:
                raise FileNotFoundError(f"No .zip archive and no extracted '{split}/{label}' folder in {gdir}")
            CLASS_DIRS[(gen, split, label)] = hits[0]
            print(f"[Drive folder] {gen}/{split}/{label} -> {hits[0]} (read directly from Drive; slower)")
        return
    print(f"\n=== {gen}: listing {archive.name} ===", flush=True)
    t0 = time.time()
    index = index_archive(archive, {(s, l) for s, l, _ in pending})
    print(f"    listed in {time.time()-t0:.0f}s", flush=True)
    for split, label, need in pending:
        if (split, label) not in index:
            raise FileNotFoundError(f"{archive.name} has no images under '{split}/{label}/'.")
        prefix, names = index[(split, label)]
        patterns, expected = build_patterns(prefix, split, label, names, need)
        out_dir = LOCAL_ROOT / gen
        class_dir = out_dir / prefix / split / label if prefix else out_dir / split / label
        print(f"-> {gen} {split}/{label}: {len(names):,} in archive, extracting about {expected:,} "
              f"(free disk {shutil.disk_usage('/content').free/1e9:.0f} GB)", flush=True)
        run_7z(archive, out_dir, patterns, class_dir, expected, f"{gen}_{split}_{label}")
        got = sum(1 for _ in class_dir.iterdir()) if class_dir.is_dir() else 0
        if got < max(1, int(expected * 0.9)):
            raise RuntimeError(f"Only {got:,} of ~{expected:,} files extracted for {gen}/{split}/{label}. "
                               f"Check /content/7z_{gen}_{split}_{label}.log (disk space? Drive quota/IO error?)")
        (LOCAL_ROOT / gen / f".done_{split}_{label}").write_text(str(class_dir))
        CLASS_DIRS[(gen, split, label)] = class_dir
        print(f"   done: {got:,} files in {class_dir}", flush=True)

# validation + test: take everything (about 6,000 per class); train: spread subset
prepare_generator(VALIDATION_GENERATOR, [("val", "nature", None), ("val", "ai", None)])
prepare_generator(TEST_GENERATOR,       [("val", "nature", None), ("val", "ai", None)])

TRAIN_QUOTAS = {
    generator: TRAIN_PER_CLASS // len(TRAIN_GENERATORS)
    + int(index < TRAIN_PER_CLASS % len(TRAIN_GENERATORS))
    for index, generator in enumerate(TRAIN_GENERATORS)
}
for generator, quota in TRAIN_QUOTAS.items():
    jobs = []
    if generator == REAL_TRAIN_GENERATOR:
        jobs.append(("train", "nature", math.ceil(TRAIN_PER_CLASS * 1.25)))
    jobs.append(("train", "ai", math.ceil(quota * 1.25)))
    prepare_generator(generator, jobs)

print("\nExtraction complete.")
for k, v in CLASS_DIRS.items():
    print(f"  {k}: {v}")

## 4. Build balanced, generator-separated ViT splits

Test (Midjourney) is selected first, then validation (ADM), then training. Train AI images are sampled evenly across the configured generator set, with real/nature images sampled from BigGAN. A global SHA-256 index excludes exact duplicate bytes across every split and rejects identical content with contradictory labels. The resulting CSV split lists are saved to Drive for reproducibility.


In [ ]:
def safe_hash(path):
    try:
        with Image.open(path) as image:
            image.verify()
        digest = hashlib.sha256()
        with open(path, "rb") as image_file:
            for chunk in iter(lambda: image_file.read(1 << 20), b""):
                digest.update(chunk)
        return digest.hexdigest()
    except (OSError, UnidentifiedImageError, Image.DecompressionBombError):
        return None


seen_hashes = {}
duplicate_counts = defaultdict(int)
unreadable_counts = defaultdict(int)


def list_images(class_dir):
    return sorted(
        str(path) for path in Path(class_dir).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def select_unique(generator, source_split, label, count, split_name):
    class_dir = CLASS_DIRS[(generator, source_split, CLASS_FOLDERS[label])]
    paths = list_images(class_dir)
    random.Random(SEED + sum(map(ord, f"{generator}:{source_split}:{label}:{split_name}"))).shuffle(paths)
    selected = []
    position = 0
    with ThreadPoolExecutor(max_workers=8) as pool:
        while len(selected) < count and position < len(paths):
            chunk = paths[position:position + 512]
            position += len(chunk)
            for image_path, digest in zip(chunk, pool.map(safe_hash, chunk)):
                if digest is None:
                    unreadable_counts[split_name] += 1
                    continue
                prior = seen_hashes.get(digest)
                if prior is not None:
                    if prior[1] != int(label):
                        raise ValueError(f"Identical image content has conflicting labels: {image_path}")
                    duplicate_counts[split_name] += 1
                    continue
                seen_hashes[digest] = (split_name, int(label))
                selected.append({
                    "path": image_path,
                    "label": int(label),
                    "generator": generator,
                    "source_split": source_split,
                    "sha256": digest,
                })
                if len(selected) == count:
                    break
    if len(selected) < count:
        print(
            f"[Warning] {split_name}/{generator}/{CLASS_FOLDERS[label]}: "
            f"requested {count:,}; found {len(selected):,} unique readable images."
        )
    return pd.DataFrame(selected, columns=["path", "label", "generator", "source_split", "sha256"])


def balanced(frame, split_name):
    counts = frame["label"].value_counts()
    if set(counts.index) != {0, 1}:
        raise RuntimeError(f"Split {split_name} is missing a class: {counts.to_dict()}")
    per_class = int(counts.min())
    parts = [frame[frame.label == label].sample(n=per_class, random_state=SEED) for label in (0, 1)]
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)


def build_single_generator_split(generator, source_split, per_class, split_name):
    parts = [select_unique(generator, source_split, label, per_class, split_name) for label in (0, 1)]
    return balanced(pd.concat(parts, ignore_index=True), split_name)


test_frame = build_single_generator_split(TEST_GENERATOR, "val", TEST_PER_CLASS, "test")
validation_frame = build_single_generator_split(
    VALIDATION_GENERATOR, "val", VALIDATION_PER_CLASS, "validation"
)

train_real = select_unique(
    REAL_TRAIN_GENERATOR, "train", 0, TRAIN_PER_CLASS, "train"
)
train_ai_parts = [
    select_unique(generator, "train", 1, quota, "train")
    for generator, quota in TRAIN_QUOTAS.items()
]
train_frame = balanced(pd.concat([train_real, *train_ai_parts], ignore_index=True), "train")

for name, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame)):
    counts = frame.label.value_counts().sort_index().to_dict()
    generators = frame.generator.value_counts().to_dict()
    print(f"{name:<10} {len(frame):>7,} images | classes={counts} | sources={generators}")
    frame.to_csv(SPLITS_DIR / f"{name}.csv", index=False)

assert not (set(train_frame.path) & set(validation_frame.path))
assert not (set(train_frame.path) & set(test_frame.path))
assert not (set(validation_frame.path) & set(test_frame.path))
assert set(train_frame.label.unique()) == set(validation_frame.label.unique()) == set(test_frame.label.unique()) == {0, 1}
assert set(validation_frame.generator.unique()) == {VALIDATION_GENERATOR}
assert set(test_frame.generator.unique()) == {TEST_GENERATOR}
assert not ({VALIDATION_GENERATOR, TEST_GENERATOR} & set(train_frame.generator.unique()))
print(f"Removed exact duplicates: {dict(duplicate_counts)}; skipped unreadable files: {dict(unreadable_counts)}")
signature_rows = pd.concat([
    frame.assign(dataset_split=split_name)[["dataset_split", "generator", "label", "sha256"]]
    for split_name, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame))
], ignore_index=True).sort_values(["dataset_split", "generator", "label", "sha256"])
SPLIT_SIGNATURE = hashlib.sha256(signature_rows.to_csv(index=False).encode("utf-8")).hexdigest()
print(f"Selected-data signature: {SPLIT_SIGNATURE}")
sync_to_drive()


## 5. Continue fine-tuning only the Hugging Face ViT

The model starts from the v3 ViT and matching image processor. Training uses the multi-generator balanced split; ADM validation selects the checkpoint, while Midjourney is used only for the final report. Mixed precision, gradient accumulation, learning-rate warmup/cosine decay, gradient clipping, early stopping, and Drive checkpoints make the run more stable and resumable. If validation does not improve, the starting ViT is retained.


In [ ]:
cached_selection = (
    json.loads(VIT_STAGE_FILE.read_text(encoding="utf-8"))
    if VIT_STAGE_FILE.is_file()
    else {}
)
vit_done = (
    VIT_SCORES_FILE.is_file()
    and VIT_TRUE_FILE.is_file()
    and (VIT_OUTPUT_DIR / "model.safetensors").is_file()
    and cached_selection.get("split_signature") == SPLIT_SIGNATURE
    and not FORCE_RETRAIN
)

if vit_done:
    vit_selection = json.loads(VIT_STAGE_FILE.read_text(encoding="utf-8"))
    vit_history_path = METRICS_DIR / "huggingface_vit_training_history.json"
    vit_history = json.loads(vit_history_path.read_text(encoding="utf-8"))
    vit_test_scores = np.load(VIT_SCORES_FILE)
    vit_test_true = np.load(VIT_TRUE_FILE)
    print("ViT stage already completed; loaded the saved checkpoint and test scores.")
else:
    processor = AutoImageProcessor.from_pretrained(str(VIT_CHECKPOINT_DIR), local_files_only=True)
    vit_model = AutoModelForImageClassification.from_pretrained(
        str(VIT_CHECKPOINT_DIR), local_files_only=True, use_safetensors=True
    )
    id2label = {int(index): str(label).strip().upper() for index, label in vit_model.config.id2label.items()}
    real_ids = [index for index, label in id2label.items() if label == "REAL"]
    fake_ids = [index for index, label in id2label.items() if label == "FAKE"]
    if len(real_ids) != 1 or len(fake_ids) != 1:
        raise ValueError(f"Expected exactly one REAL and one FAKE output class; got {id2label}")
    REAL_ID, FAKE_ID = real_ids[0], fake_ids[0]
    label_to_class = {0: REAL_ID, 1: FAKE_ID}
    print(f"ViT classes={id2label}; target mapping={label_to_class}")

    class GenImageDataset(torch.utils.data.Dataset):
        def __init__(self, frame):
            self.paths = frame.path.tolist()
            self.labels = frame.label.to_numpy(dtype=np.int64)

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, index):
            image_path = self.paths[index]
            try:
                with Image.open(image_path) as image:
                    pixels = processor(
                        images=image.convert("RGB"), return_tensors="pt"
                    )["pixel_values"].squeeze(0)
            except (OSError, UnidentifiedImageError) as error:
                raise RuntimeError(f"Cannot decode GenImage file {image_path}: {error}") from error
            return pixels, label_to_class[int(self.labels[index])]

    def collate_images(batch):
        pixel_values, labels = zip(*batch)
        return torch.stack(pixel_values), torch.tensor(labels, dtype=torch.long)

    worker_count = max(2, min(4, os.cpu_count() or 2))

    def make_loader(frame, shuffle):
        return torch.utils.data.DataLoader(
            GenImageDataset(frame),
            batch_size=VIT_BATCH_SIZE,
            shuffle=shuffle,
            num_workers=worker_count,
            pin_memory=True,
            collate_fn=collate_images,
            persistent_workers=True,
            prefetch_factor=2,
        )

    train_loader = make_loader(train_frame, True)
    validation_loader = make_loader(validation_frame, False)
    test_loader = make_loader(test_frame, False)
    vit_model.to(DEVICE)
    use_amp = DEVICE.type == "cuda"

    def evaluate_vit(model, loader, description):
        model.eval()
        truth, scores = [], []
        loss_sum, sample_count = 0.0, 0
        with torch.inference_mode():
            for pixel_values, class_ids in tqdm(loader, desc=description, leave=False):
                pixel_values = pixel_values.to(DEVICE, non_blocking=True)
                class_ids_device = class_ids.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    output = model(pixel_values=pixel_values, labels=class_ids_device)
                probabilities = torch.softmax(output.logits.float(), dim=-1)[:, FAKE_ID]
                truth.extend(0 if int(value) == REAL_ID else 1 for value in class_ids)
                scores.extend(probabilities.cpu().tolist())
                loss_sum += float(output.loss.item()) * len(class_ids)
                sample_count += len(class_ids)
        scores = np.asarray(scores, dtype=np.float64)
        labels = np.asarray(truth, dtype=np.int64)
        accuracy = float(accuracy_score(labels, (scores >= 0.5).astype(np.int64)))
        return accuracy, loss_sum / sample_count, labels, scores

    starting_accuracy, starting_loss, _, _ = evaluate_vit(vit_model, validation_loader, "ADM validation (start)")
    best_accuracy, best_loss = starting_accuracy, starting_loss
    print(f"Starting ADM validation: accuracy={starting_accuracy:.2%}, loss={starting_loss:.5f}")

    optimizer = torch.optim.AdamW(
        vit_model.parameters(), lr=VIT_LEARNING_RATE, weight_decay=VIT_WEIGHT_DECAY
    )
    updates_per_epoch = math.ceil(len(train_loader) / VIT_GRAD_ACCUMULATION_STEPS)
    total_steps = updates_per_epoch * VIT_EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(total_steps * 0.05)),
        num_training_steps=total_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    vit_history, selected_continued, bad_epochs = [], False, 0
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(VIT_EPOCHS):
        vit_model.train()
        epoch_loss, sample_count = 0.0, 0
        progress = tqdm(train_loader, desc=f"ViT epoch {epoch + 1}/{VIT_EPOCHS}")
        for batch_index, (pixel_values, class_ids) in enumerate(progress):
            pixel_values = pixel_values.to(DEVICE, non_blocking=True)
            class_ids = class_ids.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                output = vit_model(pixel_values=pixel_values, labels=class_ids)
                loss = output.loss / VIT_GRAD_ACCUMULATION_STEPS
            scaler.scale(loss).backward()
            is_update = (
                (batch_index + 1) % VIT_GRAD_ACCUMULATION_STEPS == 0
                or batch_index + 1 == len(train_loader)
            )
            if is_update:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(vit_model.parameters(), VIT_MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            epoch_loss += float(output.loss.item()) * len(class_ids)
            sample_count += len(class_ids)
            progress.set_postfix(loss=f"{epoch_loss / sample_count:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

        validation_accuracy, validation_loss, _, _ = evaluate_vit(
            vit_model, validation_loader, f"ADM validation (epoch {epoch + 1})"
        )
        record = {
            "epoch": epoch + 1,
            "train_loss": epoch_loss / sample_count,
            "validation_accuracy": validation_accuracy,
            "validation_loss": validation_loss,
            "learning_rate": scheduler.get_last_lr()[0],
        }
        vit_history.append(record)
        print(record)
        (METRICS_DIR / "huggingface_vit_training_history.json").write_text(
            json.dumps(vit_history, indent=2), encoding="utf-8"
        )
        if validation_accuracy > best_accuracy or (
            validation_accuracy == best_accuracy and validation_loss < best_loss
        ):
            best_accuracy, best_loss = validation_accuracy, validation_loss
            selected_continued, bad_epochs = True, 0
            vit_model.save_pretrained(VIT_OUTPUT_DIR, safe_serialization=True)
            processor.save_pretrained(VIT_OUTPUT_DIR)
            print("Saved new validation-best ViT checkpoint and syncing to Drive.")
            sync_to_drive()
        else:
            bad_epochs += 1
            if bad_epochs >= VIT_EARLY_STOPPING_PATIENCE:
                print("Early stopping: ADM validation did not improve.")
                break

    if not selected_continued:
        print("No validation improvement; preserving the v3 starting ViT checkpoint.")
        shutil.copytree(VIT_CHECKPOINT_DIR, VIT_OUTPUT_DIR, dirs_exist_ok=True)

    vit_selection = {
        "starting_validation_accuracy": starting_accuracy,
        "starting_validation_loss": starting_loss,
        "selected_validation_accuracy": best_accuracy,
        "selected_validation_loss": best_loss,
        "continued_checkpoint_selected": bool(selected_continued),
        "validation_generator": VALIDATION_GENERATOR,
        "split_signature": SPLIT_SIGNATURE,
    }
    VIT_STAGE_FILE.write_text(json.dumps(vit_selection, indent=2), encoding="utf-8")
    del vit_model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

    selected_model = AutoModelForImageClassification.from_pretrained(
        str(VIT_OUTPUT_DIR), local_files_only=True, use_safetensors=True
    ).to(DEVICE)
    _, _, vit_test_true, vit_test_scores = evaluate_vit(
        selected_model, test_loader, f"Held-out {TEST_GENERATOR} test"
    )
    np.save(VIT_SCORES_FILE, vit_test_scores)
    np.save(VIT_TRUE_FILE, vit_test_true)
    print(f"Saved ViT and processor: {VIT_OUTPUT_DIR}")
    del selected_model, train_loader, validation_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()
    sync_to_drive()


In [ ]:
def vit_binary_metrics(y_true, ai_scores):
    y_true = np.asarray(y_true, dtype=np.int64)
    ai_scores = np.asarray(ai_scores, dtype=np.float64)
    if len(y_true) != len(ai_scores) or not np.isfinite(ai_scores).all():
        raise ValueError("Invalid ViT test labels or scores.")
    predictions = (ai_scores >= 0.5).astype(np.int64)
    return {
        "n": int(len(y_true)),
        "real_count": int(np.sum(y_true == 0)),
        "ai_generated_count": int(np.sum(y_true == 1)),
        "threshold": 0.5,
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1_score": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, ai_scores)),
        "confusion_matrix_labels_0_1": confusion_matrix(y_true, predictions, labels=[0, 1]).tolist(),
    }


expected_test_labels = test_frame.label.to_numpy(dtype=np.int64)
if len(vit_test_true) != len(test_frame) or not np.array_equal(vit_test_true, expected_test_labels):
    raise RuntimeError("Saved ViT test outputs do not match the current held-out test split.")
vit_results = vit_binary_metrics(vit_test_true, vit_test_scores)
comparison = pd.DataFrame([{"model": "Fine-tuned Hugging Face ViT", **vit_results}])
display(comparison.style.format({
    "accuracy": "{:.2%}", "precision": "{:.2%}", "recall": "{:.2%}",
    "f1_score": "{:.2%}", "roc_auc": "{:.4f}",
}))
print("Confusion matrix [[real->real, real->AI], [AI->real, AI->AI]]:")
print(vit_results["confusion_matrix_labels_0_1"])

summary = {
    "dataset": "GenImage official benchmark",
    "model": "Fine-tuned Hugging Face ViT",
    "label_mapping": {"nature/real": 0, "ai-generated": 1},
    "generator_partition": {
        "training_generators": TRAIN_GENERATORS,
        "real_training_generator": REAL_TRAIN_GENERATOR,
        "validation_generator": VALIDATION_GENERATOR,
        "test_generator": TEST_GENERATOR,
    },
    "requested_images_per_class": {
        "train": TRAIN_PER_CLASS,
        "validation": VALIDATION_PER_CLASS,
        "test": TEST_PER_CLASS,
    },
    "actual_split_sizes": {
        "train": int(len(train_frame)),
        "validation": int(len(validation_frame)),
        "test": int(len(test_frame)),
    },
    "training_samples_by_generator": {
        str(generator): int(count)
        for generator, count in train_frame.generator.value_counts().items()
    },
    "starting_vit_checkpoint": str(VIT_CHECKPOINT_DIR),
    "vit_checkpoint_selection": vit_selection,
    "results": {"huggingface_vit": vit_results},
    "warning": (
        f"Test results measure transfer to the held-out {TEST_GENERATOR} generator only. "
        "Exact byte-identical duplicates were excluded, but visually similar or re-encoded "
        "duplicates may remain. This is not a universal real-world accuracy estimate. "
        "Do not adjust training based on this test score."
    ),
}
(METRICS_DIR / "vit_genimage_v4_metrics.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)
comparison.to_csv(METRICS_DIR / "vit_genimage_v4_metrics.csv", index=False)
sync_to_drive()
print(f"ViT metrics saved under {METRICS_DIR}")


## 6. Export only the selected ViT, processor, splits, and metrics

The output ZIP is `ai_vs_real_vit_genimage_outputs_v4.zip` in My Drive and downloads at the end if enabled. It contains no CNN checkpoint. Test results are held out and should be reported as generator-specific performance, not used to select training settings.


In [ ]:
sync_to_drive()
zip_path = Path(shutil.make_archive(
    "/content/ai_vs_real_vit_genimage_outputs_v4", "zip", OUTPUT_DIR
))
print(f"Created {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MiB)")
shutil.copy2(zip_path, DRIVE_OUT / zip_path.name)
print(f"Copied ZIP to Drive: {DRIVE_OUT / zip_path.name}")
if DOWNLOAD_ZIP_AT_END:
    files.download(str(zip_path))
